# 07. 오픈소스에서 배우는 Jev 패턴 (1) 메모리 압축

## 이 시리즈에 대해

01~06번에서는 Jev 를 고객지원 에이전트 하나에 붙였습니다. 07~10번은 다른 사람들이 Jev 를 어디에 쓰고 있는지 살펴봅니다. 출발점은 블로그 글 [Jev 활용 프로젝트 정리](https://javaexpert.tistory.com/1840)에 소개된 오픈소스 8개입니다.

그 글의 주장은 한 줄로 줄일 수 있습니다. 에이전트가 하는 일을 **생성**과 **판단**으로 나누고, 큰 LLM 에는 계획과 추론만 맡기고, 반복되는 작은 판단은 판단 전용 모델에 맡기자는 것입니다.

| 프로젝트 | Jev 가 맡는 판단 | 이 튜토리얼에서 |
|---|---|---|
| [fast-jev-compaction](https://github.com/tamaratran/fast-jev-compaction) | 대화 기록에서 무엇을 남기고 지울지 | **07번 (이 노트북)** |
| [pi-jev](https://github.com/y0usaf/pi-jev) | 도구 실행 전 위험 판정, 실행 후 출력 판정 | 08번 |
| [jev-search](https://github.com/superagents-lab/jev-search) | 검색 소스 선택, 결과 관련도 | 09번 |
| [hermes-jev-skills](https://github.com/kerpopule/hermes-jev-skills) | 모델 라우팅, 메시지 분류 등 여러 판단 지점 | 09번 |
| [jev-browser](https://github.com/Ying-Kai-Liao/jev-browser) | 화면에서 어느 요소를 누를지, 단계가 끝났는지 | 10번 |
| [NanoJev](https://github.com/TianyuCodings/NanoJev) | 작은 모델(0.6B)로 probability 분포를 내는 판단 모델 구현 | 10번 끝에서 소개만 |
| [open-alternative-jev](https://github.com/ikermoel/open-alternative-jev) | 공개 가중치 모델로 같은 방식 재현 | 10번 끝에서 소개만 |
| [Reticle](https://github.com/reticlehq/reticle) | 에이전트가 만든 코드의 실행 중 검증 (Jev 연동은 계획 단계) | 10번 끝에서 소개만 |

각 노트북의 코드는 해당 프로젝트의 README 에 적힌 방식을 **참고해서 이 저장소에 새로 작성한 것**입니다. 원 프로젝트의 코드를 가져온 것이 아니고, 기준값 같은 숫자를 인용할 때는 출처가 README 임을 밝힙니다.

같은 내용을 화면에서 눌러 볼 수 있습니다. 앱 왼쪽의 "오픈소스 패턴" 탭들이 이 노트북들과 같은 모듈(`jev_agent/patterns/`)을 씁니다.

## 배경

Claude Code 같은 코딩 에이전트는 오래 실행됩니다. 파일을 읽고, 명령을 실행하고, 그 결과가 전부 대화 기록에 쌓입니다. 기록이 컨텍스트 한도에 가까워지면 줄여야 합니다.

흔한 방법은 LLM 에게 요약을 시키는 것입니다. 이 방법은 느리고, 요약문이 원문과 달라질 수 있습니다. 파일 경로가 바뀌거나 오류 메시지의 숫자가 틀리게 옮겨지면 에이전트는 틀린 기억으로 일하게 됩니다.

fast-jev-compaction 은 요약하지 않습니다. 도구 호출마다 **남길지, 줄일지, 지울지**만 정합니다. 남은 것은 원문 그대로입니다.

## 직관

이삿짐을 쌀 때와 같습니다. 물건마다 설명서를 새로 쓰지 않습니다. 상자에 넣을지, 일부만 챙길지, 버릴지만 정합니다.

도구 호출 하나에 예/아니오 질문 두 개를 던집니다.

| 질문 | 뜻 |
|---|---|
| `keep_call` | 이 호출이 있었다는 사실을 알아야 하는가 |
| `keep_result` | 결과 원문을 그대로 다시 봐야 하는가 |

| 답 | 처리 |
|---|---|
| `keep_result` 가 기준 이상 | **KEEP** 호출과 결과를 그대로 둔다 |
| `keep_call` 만 기준 이상 | **TRUNCATE** 호출은 두고 결과는 앞부분만 남긴다 |
| 둘 다 기준 미만 | **DROP** 호출과 결과를 모두 지운다 |

기준값은 0.5 입니다 (원 프로젝트 README 의 기본값).

In [1]:
import sys
sys.path.insert(0, "..")  # 프로젝트 루트의 jev_agent 패키지를 불러오기 위함

import pandas as pd
from dotenv import load_dotenv
load_dotenv("../.env")

from jev_agent.jev import JevClient, choice, noul, score
from jev_agent.patterns import run_pattern

pd.set_option("display.max_colwidth", 60)
jev = JevClient()

In [2]:
def rows_table(output: dict) -> pd.DataFrame:
    """패턴 결과의 rows 를 표로 바꾼다. probability 는 큰 순서로 세 개까지 한 칸에 적는다."""
    records = []
    for item in output["rows"]:
        top = sorted(item["probabilities"].items(), key=lambda pair: -pair[1])[:3]
        records.append({
            "항목": item["title"],
            "판정": item["verdict"],
            "probability": ", ".join(f"{name} {value:.2f}" for name, value in top),
        })
    return pd.DataFrame(records)

## 예시 대화

로그인 버그를 고치는 코딩 에이전트의 기록입니다. 도구 호출이 7번 있고, 그중에는 목표와 무관한 CSS 파일 읽기와 긴 설치 로그가 섞여 있습니다.

In [3]:
from jev_agent.patterns import compaction

print("목표:", compaction.SAMPLE_GOAL, "\n")
for entry in compaction.SAMPLE_TRANSCRIPT:
    if entry["type"] == "tool":
        print(f"[도구 {entry['id']}] {entry['name']}({entry['args']})  → 결과 {len(entry['result']):,}자")
    else:
        print(f"[{entry['role']}] {entry['content']}")

목표: 로그인 API 가 500 을 반환하는 버그를 고치고 테스트를 통과시킨다 

[user] 로그인하면 500 이 나요. 고쳐 주세요.
[도구 t1] list_files({'path': '.'})  → 결과 342자
[도구 t2] read_file({'path': 'static/theme.css'})  → 결과 1,440자
[도구 t3] grep({'pattern': 'def login', 'path': 'src/'})  → 결과 55자
[도구 t4] read_file({'path': 'src/auth/routes.py'})  → 결과 227자
[도구 t5] run_shell({'command': 'pip install -e .'})  → 결과 1,807자
[도구 t6] run_tests({'path': 'tests/test_login.py'})  → 결과 144자
[assistant] user 가 None 일 때의 처리가 빠져 있습니다.
[도구 t7] edit_file({'path': 'src/auth/routes.py', 'change': 'user 가 None 이면 401 반환'})  → 결과 31자


## Jev 에 무엇을 보내는가

`plan()` 이 Jev 에 보낼 `state` 와 `questions` 를 만듭니다. 눈여겨볼 점은 **도구 결과 원문을 보내지 않는다**는 것입니다. 결과는 길이만 적은 한 줄로 바뀝니다. 그래서 대화가 아무리 길어도 Jev 에 보내는 양은 작습니다.

In [4]:
payload = {"goal": compaction.SAMPLE_GOAL, "transcript": compaction.SAMPLE_TRANSCRIPT}
state, questions = compaction.plan(payload)

print("state['conversation']:")
for line in state["conversation"]:
    print("  ", line)
print(f"\n질문 {len(questions)}개 (도구 호출 {len(questions) // 2}개 × 2). 예:")
print("  call_t4  :", questions["call_t4"]["instructions"])
print("  result_t4:", questions["result_t4"]["instructions"])

state['conversation']:
   [user] 로그인하면 500 이 나요. 고쳐 주세요.
   [tool t1] list_files({'path': '.'}) -> ok, 342 chars (omitted)
   [tool t2] read_file({'path': 'static/theme.css'}) -> ok, 1440 chars (omitted)
   [tool t3] grep({'pattern': 'def login', 'path': 'src/'}) -> ok, 55 chars (omitted)
   [tool t4] read_file({'path': 'src/auth/routes.py'}) -> ok, 227 chars (omitted)
   [tool t5] run_shell({'command': 'pip install -e .'}) -> ok, 1807 chars (omitted)
   [tool t6] run_tests({'path': 'tests/test_login.py'}) -> ok, 144 chars (omitted)
   [assistant] user 가 None 일 때의 처리가 빠져 있습니다.
   [tool t7] edit_file({'path': 'src/auth/routes.py', 'change': 'user 가 None 이면 401 반환'}) -> ok, 31 chars (omitted)

질문 14개 (도구 호출 7개 × 2). 예:
  call_t4  : goal 에 적힌 목표를 이루려면 도구 호출 t4 이 있었다는 사실을 알아야 하는가?
  result_t4: goal 에 적힌 목표를 이루려면 도구 호출 t4 의 결과 원문을 다시 읽어야 하는가?


## 실행

질문 14개가 호출 한 번에 처리됩니다.

In [5]:
output = run_pattern(jev, "compaction", payload)

print({item["label"]: item["value"] for item in output["stats"]}, f"| {output['latency_ms']}ms | ${output['cost']:.6f}")
pd.DataFrame([
    {"도구 호출": item["title"], "판정": item["verdict"],
     "keep_call": round(item["probabilities"]["keep_call"], 2),
     "keep_result": round(item["probabilities"]["keep_result"], 2)}
    for item in output["rows"]
])

{'압축 전': '4,098자', '압축 후': '498자', '절감': '88%', '질문 수': 14} | 267ms | $0.000087


,도구 호출,판정,keep_call,keep_result
0,list_files({'path': '.'}),DROP,0.22,0.09
1,read_file({'path': 'static/theme.css'}),DROP,0.08,0.06
2,"grep({'pattern': 'def login', 'path': 'src/'})",TRUNCATE,0.50,0.22
3,read_file({'path': 'src/auth/routes.py'}),KEEP,0.71,0.54
4,run_shell({'command': 'pip install -e .'}),DROP,0.32,0.10
5,run_tests({'path': 'tests/test_login.py'}),TRUNCATE,0.78,0.49
6,"edit_file({'path': 'src/auth/routes.py', 'change': 'user...",KEEP,0.89,0.64


## 실제로 줄이기

판단대로 대화를 줄입니다. KEEP 은 그대로, TRUNCATE 는 결과 앞 120자만, DROP 은 통째로 사라집니다.

In [6]:
compacted = compaction.compact(compaction.SAMPLE_TRANSCRIPT, output["decisions"])

for entry in compacted:
    if entry["type"] == "tool":
        print(f"[도구 {entry['id']}] {entry['name']} → {entry['result'][:150]!r}")
    else:
        print(f"[{entry['role']}] {entry['content']}")

[user] 로그인하면 500 이 나요. 고쳐 주세요.
[도구 t3] grep → 'src/auth/routes.py:41:def login(payload: LoginRequest):'
[도구 t4] read_file → 'def login(payload: LoginRequest):\n    user = repo.find_by_email(payload.email)\n    if not verify(payload.password, user.password_hash):  # user 가 None'
[도구 t6] run_tests → "FAILED tests/test_login.py::test_unknown_email_returns_401\nAttributeError: 'NoneType' object has no attribute 'password_ …(이하 24자 생략)"
[assistant] user 가 None 일 때의 처리가 빠져 있습니다.
[도구 t7] edit_file → '1 file changed, 2 insertions(+)'


## 목표가 바뀌면 판단도 바뀐다

같은 대화에 목표만 "CSS 여백 정리"로 바꿉니다. 앞에서 버렸던 CSS 파일 읽기가 이번에는 중요해지는지 확인합니다.

In [7]:
css_output = run_pattern(
    jev, "compaction", {"goal": "버튼과 카드의 CSS 여백을 정리한다", "transcript": compaction.SAMPLE_TRANSCRIPT}
)
pd.DataFrame({
    "도구 호출": [item["title"] for item in output["rows"]],
    "목표: 로그인 버그": [item["verdict"] for item in output["rows"]],
    "목표: CSS 정리": [item["verdict"] for item in css_output["rows"]],
})

,도구 호출,목표: 로그인 버그,목표: CSS 정리
0,list_files({'path': '.'}),DROP,DROP
1,read_file({'path': 'static/theme.css'}),DROP,KEEP
2,"grep({'pattern': 'def login', 'path': 'src/'})",TRUNCATE,DROP
3,read_file({'path': 'src/auth/routes.py'}),KEEP,DROP
4,run_shell({'command': 'pip install -e .'}),DROP,DROP
5,run_tests({'path': 'tests/test_login.py'}),TRUNCATE,DROP
6,"edit_file({'path': 'src/auth/routes.py', 'change': 'user...",KEEP,DROP


## 정리

- 요약문을 쓰지 않고 원문을 남길지 말지만 정합니다. 남은 내용은 원문 그대로라 사실이 바뀌지 않습니다.
- Jev 는 도구 결과 원문을 보지 않고, 호출의 이름과 인자, 결과 길이, 대화 흐름만 보고 판단합니다. 그래서 입력이 작고 빠릅니다.
- 같은 이유로 한계도 있습니다. 결과 안에 무엇이 들어 있는지 모르므로, 이름만 봐서는 중요해 보이지 않는 호출의 결과를 버릴 수 있습니다. 위 표에서 KEEP 으로 나온 항목들의 `keep_result` 가 0.5 를 크게 넘지 못한다면 경계에 가까운 판단이라는 뜻입니다.
- 실제로 쓸 때는 사용자의 첫 요청이나 최근 몇 턴처럼 무조건 남길 항목을 따로 고정해 둡니다. 원 프로젝트도 고정 항목은 질문에서 빼 둡니다.